___
<h1> Machine Learning </h1>
<h2> Systems Engineering and Computer Technologies / Engenharia de Sistemas e Tecnologias Informáticas
(LESTI)</h2>
<h3> Instituto Superior de Engenharia / Universidade do Algarve </h3>

[LESTI](https://ise.ualg.pt/curso/1941) / [ISE](https://ise.ualg.pt) / [UAlg](https://www.ualg.pt)

Pedro J. S. Cardoso (pcardoso@ualg.pt)

___

# $k$-Nearest Neighbors (kNN): Regression

In [ ]:
from sklearn.model_selection import train_test_split
# !pip3 install mglearn

import matplotlib.pyplot as plt
#import mglearn.plots

from sklearn.neighbors import KNeighborsRegressor
import numpy as np

np.random.seed(1)

## k-Neighbors Regression

There is also a regression variant of the k-nearest neighbors algorithm. Again, let’s start by using a single nearest neighbor, this time using the wave dataset. 

The prediction using a single neighbor (1-NN) is just the target value of the nearest neighbor. When using multiple nearest neighbors for regression, the prediction is the average (or mean) of the relevant neighbors

In [ ]:
np.random.seed(2)

n_samples = 50
a, b = -10, 10

# set of "x's"
timeline = np.linspace(a, b, n_samples).reshape(-1, 1)

# set a 'wave'
def make_wave(timeline):
    return 10 * np.cos(timeline) + 5 * np.random.rand(len(timeline))

wave = make_wave(timeline.T[0])

In [ ]:
fig = plt.figure(figsize=(15,5))
ax = fig.add_subplot(111)
ax.plot(timeline, wave, linestyle='None', marker='o')
plt.show()

As usual, split the test

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(timeline, wave, test_size=.1 )

In [ ]:
fig = plt.figure(figsize=(15,5))
ax = fig.add_subplot(111)
ax.plot(x_train, y_train, linestyle="none", marker='o', c="red")
ax.plot(x_test, y_test, linestyle="none", marker='o', c="blue")
plt.show()

and then train the model

In [ ]:
knr = KNeighborsRegressor(
        n_neighbors=3, 
        weights='distance'
)

knr.fit(x_train, y_train)

And now the predictions

In [ ]:
print(x_test)
print(y_test)
print(knr.predict(x_test))

And we can see compute the errors

In [ ]:
y_pred = knr.predict(x_test)

delta = y_pred - y_test
delta

And the score (coefficient of determination $R^2$ of the prediction) is

In [ ]:
knr.score(x_test, y_test)

In [ ]:
fig = plt.figure(figsize=(15,5))
ax = fig.add_subplot(111)
ax.plot(x_train, y_train, linestyle="none", marker='o', c="red")
ax.plot(x_test, y_test, linestyle="none", marker='o', c="blue")
ax.plot(x_test, y_pred, linestyle="none", marker='o', c="green")
ax.legend(["train", "test", "pred"])
ax.set_title("KNeighborsRegressor with 3 neighbors and distance weighting")
plt.show()

In [ ]:
X = np.arange(a, b, .01).reshape(-1, 1)

K = 30
fig = plt.figure(figsize=(10 * 2, 5 * K))

im_idx = 0

for nn in range(1, K, 2):
    #  weights='uniform'
    # ‘uniform’ : uniform weights. All points in each neighborhood are weighted equally.
    knr = KNeighborsRegressor(n_neighbors=nn, weights='uniform').fit(x_train, y_train)
    y = knr.predict(X)

    im_idx += 1
    ax = fig.add_subplot(K - 1, 2, im_idx)
    ax.plot(timeline, wave, label='original')
    ax.plot(x_train, y_train, linestyle="none", marker='o', c="red")
    ax.plot(X, y, label=f'nn = {nn}, weights = uniform, r2 = {knr.score(x_test, y_test)}')
    ax.legend()

    # weights='distance'
    # ‘distance’ : weight points by the inverse of their distance. In this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.
    knr = KNeighborsRegressor(n_neighbors=nn, weights='distance').fit(x_train, y_train)
    y = knr.predict(X)
    
    im_idx += 1
    ax = fig.add_subplot(K - 1, 2, im_idx)
    ax.plot(timeline, wave, label='original')
    ax.plot(X, y, label=f'nn = {nn}, weights = distance,  r2= {knr.score(x_test, y_test)} ')
    ax.plot(x_train, y_train, linestyle="none", marker='o', c="red")
    ax.legend()
    
plt.show()

Let us see how it behaves outside the samples

In [ ]:
# supposing a<0
X = np.arange(2 * a, 2 * b, .1).reshape(-1, 1)

K = 15
fig = plt.figure(figsize=(10 * 2, 5 * K))

im_idx = 0

for nn in range(1, K, 2):
    #  weights='uniform'
    # ‘uniform’ : uniform weights. All points in each neighborhood are weighted equally.
    knr = KNeighborsRegressor(n_neighbors=nn, weights='uniform').fit(timeline, wave)
    y = knr.predict(X)

    im_idx += 1
    ax = fig.add_subplot(K - 1, 2, im_idx)
    ax.plot(timeline, wave, label='original')
    ax.plot(X, y, label='nn = {}'.format(nn))
    ax.legend()

    # weights='distance'
    # ‘distance’ : weight points by the inverse of their distance. In this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.
    knr = KNeighborsRegressor(n_neighbors=nn, weights='distance').fit(timeline, wave)
    y = knr.predict(X)
    
    im_idx += 1
    ax = fig.add_subplot(K - 1, 2, im_idx)
    ax.plot(timeline, wave, label='original')
    ax.plot(X, y, label='nn = {}'.format(nn))
    ax.legend()
    
plt.show()

## Example: Grid search with cross-validation on Iris using k-NN

Although this notebook focuses on regression, k-NN can also be used for classification. We split Iris into training and test sets. Five-fold cross-validation on the training set selects the hyperparameters, and the held-out test set is used only for the final evaluation. The grid explores the number of neighbors, voting weights, distance metric, Minkowski distance power, and search algorithm. Scaling is part of the pipeline, so it is fitted separately within each cross-validation fold.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.20, stratify=iris.target, random_state=1
)
print(f'Train: {len(y_train)} | Test: {len(y_test)}')

knn_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier())
])

param_grid = [
    {
        'knn__n_neighbors': [1, 3, 5, 7, 9, 11, 13, 15],
        'knn__weights': ['uniform', 'distance'],
        'knn__metric': ['minkowski'],
        'knn__p': [1, 2, 3],
        'knn__algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute']
    },
    {
        'knn__n_neighbors': [1, 3, 5, 7, 9, 11, 13, 15],
        'knn__weights': ['uniform', 'distance'],
        'knn__metric': ['chebyshev'],
        'knn__algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute']
    }
]
 
iris_search = GridSearchCV(
    estimator=knn_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=3
)
iris_search.fit(X_train, y_train)

print(f'Best parameters: {iris_search.best_params_}')
print(f'Mean cross-validation accuracy: {iris_search.best_score_:.3f}')
print(f'Test accuracy: {iris_search.score(X_test, y_test):.3f}')